In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import plotly.graph_objects as go

# 1) Load Data
transform = transforms.ToTensor()
train_data = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

class MLP(nn.Module):
    def __init__(self, hidden_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28*28, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# 3) Train Function
def train_model(hidden_size):
    model = MLP(hidden_size)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    train_losses, accuracies = [], []

    print(f"Starting training for hidden_size: {hidden_size}")
    for epoch in range(5):
        total_loss, correct, total = 0, 0, 0
        for images, labels in train_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        avg_loss = total_loss / len(train_loader)
        acc = 100 * correct / total
        train_losses.append(avg_loss)
        accuracies.append(acc)
        print(f"Epoch {epoch+1}: Loss {avg_loss:.4f}, Accuracy {acc:.2f}%")
    return train_losses, accuracies

# 4) Experiments and Plotting
loss1, acc1 = train_model(128)
loss2, acc2 = train_model(256)

fig = go.Figure()
fig.add_trace(go.Scatter(y=acc1, name='Hidden 128', mode='lines+markers'))
fig.add_trace(go.Scatter(y=acc2, name='Hidden 256', mode='lines+markers'))
fig.update_layout(title="Accuracy Comparison", xaxis_title="Epoch", yaxis_title="Accuracy %")
fig.show()

100%|██████████| 9.91M/9.91M [00:00<00:00, 18.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 498kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.61MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 1.73MB/s]


Starting training for hidden_size: 128
Epoch 1: Loss 0.3469, Accuracy 90.53%
Epoch 2: Loss 0.1595, Accuracy 95.31%
Epoch 3: Loss 0.1097, Accuracy 96.77%
Epoch 4: Loss 0.0820, Accuracy 97.65%
Epoch 5: Loss 0.0652, Accuracy 98.03%
Starting training for hidden_size: 256
Epoch 1: Loss 0.3011, Accuracy 91.61%
Epoch 2: Loss 0.1278, Accuracy 96.28%
Epoch 3: Loss 0.0844, Accuracy 97.50%
Epoch 4: Loss 0.0615, Accuracy 98.17%
Epoch 5: Loss 0.0465, Accuracy 98.60%


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import plotly.graph_objects as go
import pandas as pd

# 1) Preprocessing (Normalization)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Partitioning
train_data = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_data = datasets.MNIST(root='./data', train=False, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# 2) MLP Architecture
class MLP(nn.Module):
    def __init__(self, hidden_size):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(28*28, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, 10)

    def forward(self, x):
        x = x.view(-1, 28*28)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 3) Training and Evaluation function
def run_experiment(hidden_size):
    model = MLP(hidden_size)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

    print(f"Starting Experiment with {hidden_size} neurons...")
    for epoch in range(5):
        model.train()
        train_loss, train_correct = 0, 0
        for images, labels in train_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            train_correct += (outputs.argmax(1) == labels).sum().item()

        model.eval()
        test_loss, test_correct = 0, 0
        with torch.no_grad():
            for images, labels in test_loader:
                outputs = model(images)
                test_loss += criterion(outputs, labels).item()
                test_correct += (outputs.argmax(1) == labels).sum().item()

        history['test_acc'].append(100. * test_correct / len(test_loader.dataset))
        history['test_loss'].append(test_loss/len(test_loader))
        print(f"Epoch {epoch+1}: Test Acc {history['test_acc'][-1]:.2f}%")
    return history

# 4) Experiments
res128 = run_experiment(128)
res256 = run_experiment(256)

# 5) Visualization
fig = go.Figure()
fig.add_trace(go.Scatter(y=res128['test_acc'], name='128 Neurons'))
fig.add_trace(go.Scatter(y=res256['test_acc'], name='256 Neurons'))
fig.update_layout(title="Accuracy Comparison", xaxis_title="Epoch", yaxis_title="Accuracy %")
fig.show()

# Final Comparison Table
print("\n--- Project Results Comparison ---")
df = pd.DataFrame({
    'Neurons': [128, 256],
    'Final Test Acc': [f"{res128['test_acc'][-1]:.2f}%", f"{res256['test_acc'][-1]:.2f}%"],
    'Final Test Loss': [f"{res128['test_loss'][-1]:.4f}", f"{res256['test_loss'][-1]:.4f}"]
})
print(df.to_string(index=False))

Starting Experiment with 128 neurons...
Epoch 1: Test Acc 95.37%
Epoch 2: Test Acc 97.17%
Epoch 3: Test Acc 97.10%
Epoch 4: Test Acc 97.17%
Epoch 5: Test Acc 97.64%
Starting Experiment with 256 neurons...
Epoch 1: Test Acc 96.22%
Epoch 2: Test Acc 97.05%
Epoch 3: Test Acc 97.66%
Epoch 4: Test Acc 97.49%
Epoch 5: Test Acc 97.53%



--- Project Results Comparison ---
 Neurons Final Test Acc Final Test Loss
     128         97.64%          0.0791
     256         97.53%          0.0797
